In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_excel('chatgpt_style_reviews_dataset (1).xlsx')

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

text_cols = ["title", "review", "username", "version"]
df[text_cols] = df[text_cols].astype("string")

int_cols = ["rating", "helpful_votes", "review_length"]
df[int_cols] = df[int_cols].apply(pd.to_numeric, errors="coerce").astype("Int64")

cat_cols = ["platform", "language", "location"]
df[cat_cols] = df[cat_cols].astype("category")

df["verified_purchase"] = df["verified_purchase"].astype("bool")


In [ ]:
df["date_only"] = df["date"].dt.date

df["date_only"].value_counts().sort_index(ascending=False)


In [ ]:

df[df["date"].isna()]


In [ ]:
df

In [ ]:
df.drop(columns=['title','username'],inplace=True)

In [ ]:
df

In [ ]:
import re

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import spacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


In [ ]:
df['review'] = df['review'].fillna('')

In [ ]:
def map_rating(r):
    if r >= 4:
        return 2      # Positive
    elif r == 3:
        return 1      # Neutral
    else:
        return 0      # Negative

df['sentiment'] = df['rating'].apply(map_rating)

In [ ]:
# Load spacy model
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words('english'))


def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+', '', text)

    # Remove special characters & numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    doc = nlp(" ".join(tokens))
    lemmas = [token.lemma_ for token in doc]

    return " ".join(lemmas)


In [ ]:
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_review'] = df['review'].astype(str).apply(clean_text)


In [ ]:
#EDA

In [ ]:
!pip install wordcloud seaborn nltk -q


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from wordcloud import WordCloud
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
from collections import Counter
import pandas as pd
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)


In [ ]:
print("📊 Current df info:")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nRating sample:\n", df['rating'].value_counts().sort_index())
print("\nPlatform sample:", df['platform'].value_counts())
print("Verified unique:", df['verified_purchase'].unique())
print("✅ Data verified!")


In [ ]:
#1 EDA
plt.figure(figsize=(10, 6))
rating_counts = df['rating'].value_counts().sort_index()
plt.bar(rating_counts.index, rating_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
plt.title('📊 1. Distribution of Review Ratings (1-5 Stars)', fontsize=14, fontweight='bold')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(range(1,6))
for i, v in enumerate(rating_counts.values):
    plt.text(i+1, v+1, str(v), ha='center', fontweight='bold')
plt.show()
print(f"Insight: {'Mostly happy' if rating_counts[5] > rating_counts[1] else 'Polarized/mixed'} sentiment")



In [ ]:
#2
helpful = df[df['helpful_votes'] > 10].shape[0]
total = len(df)
plt.figure(figsize=(8, 8))
plt.pie([helpful, total-helpful], labels=['👍 Helpful (>10 votes)', '👎 Not Helpful'],
        autopct='%1.1f%%', colors=['#4ECDC4', '#FFEAA7'], startangle=90)
plt.title('👍👎 2. Helpful Reviews Distribution', fontsize=14, fontweight='bold')
plt.show()
print(f"Insight: {helpful} ({helpful/total*100:.1f}%) reviews found valuable by community")



In [ ]:
#3
def clean_words(text):
    if pd.isna(text): return ''
    words = re.findall(r'\b[a-zA-Z]{3,}\b', str(text).lower())
    return ' '.join([w for w in words if w not in stop_words])

pos_reviews = df[df['rating'] >= 4]['review'].apply(clean_words)
neg_reviews = df[df['rating'] <= 2]['review'].apply(clean_words)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
wc_pos = WordCloud(width=400, height=400, background_color='white', colormap='Greens').generate(' '.join(pos_reviews))
wc_neg = WordCloud(width=400, height=400, background_color='white', colormap='Reds').generate(' '.join(neg_reviews))

ax1.imshow(wc_pos, interpolation='bilinear')
ax1.axis('off')
ax1.set_title('🧭 3A. Positive Reviews (4-5 Stars)', fontsize=14, fontweight='bold')

ax2.imshow(wc_neg, interpolation='bilinear')
ax2.axis('off')
ax2.set_title('🧭 3B. Negative Reviews (1-2 Stars)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



In [ ]:
#4
df['date_clean'] = pd.to_datetime(df['date'], errors='coerce')
monthly_rating = df.dropna(subset=['date_clean']).groupby(df['date_clean'].dt.to_period('M'))['rating'].mean()
plt.figure(figsize=(12, 6))
monthly_rating.plot(kind='line', marker='o', linewidth=2, markersize=8)
plt.title('📆 4. Average Rating Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Average Rating')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.show()
print("Insight:", "Upward trend" if monthly_rating.iloc[-1] > monthly_rating.iloc[0] else "Stable/declining")


In [ ]:
#5
location_ratings = df.groupby('location')['rating'].agg(['mean', 'count']).sort_values('mean', ascending=False)
top_locations = location_ratings.head(10)
plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(top_locations)), top_locations['mean'],
               color='skyblue', alpha=0.7, width=0.6)
plt.title('🌍 5. Average Ratings by Location (Top 10)', fontsize=14, fontweight='bold')
plt.xlabel('Location')
plt.ylabel('Average Rating')
plt.xticks(range(len(top_locations)), top_locations.index, rotation=45)
for i, bar in enumerate(bars):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{top_locations["mean"].iloc[i]:.2f}', ha='center')
plt.tight_layout()
plt.show()

In [ ]:
#6
platform_stats = df.groupby('platform')['rating'].agg(['mean', 'count']).round(2)
plt.figure(figsize=(10, 6))
x = range(len(platform_stats))
plt.bar(x, platform_stats['mean'], color=['#FF9999','#66B2FF','#99FF99','#FFCC99','#FF99FF'])
plt.title('🧑‍💻 6. Average Ratings by Platform', fontsize=14, fontweight='bold')
plt.xlabel('Platform')
plt.ylabel('Average Rating')
plt.xticks(x, platform_stats.index, rotation=45)
for i, v in enumerate(platform_stats['mean']):
    plt.text(i, v + 0.05, f'{v}', ha='center', fontweight='bold')
plt.show()
print("Best platform:", platform_stats['mean'].idxmax())

In [ ]:
#7
verified_avg = df[df['verified_purchase'] == True]['rating'].mean()
non_verified_avg = df[df['verified_purchase'] == False]['rating'].mean() if (df['verified_purchase'] == False).sum() > 0 else 0
plt.figure(figsize=(8, 6))
plt.bar([' Verified', 'Non-Verified'], [verified_avg, non_verified_avg],
        color=['#4ECDC4', '#FF6B6B'], alpha=0.8, width=0.5)
plt.title(' 7. Verified vs Non-Verified User Satisfaction', fontsize=14, fontweight='bold')
plt.ylabel('Average Rating')
plt.ylim(0, 5)
for i, v in enumerate([verified_avg, non_verified_avg]):
    plt.text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')
plt.show()
print(f"Insight: Verified users {'happier' if verified_avg > non_verified_avg else 'similar/less satisfied'}")

In [ ]:
#8
plt.figure(figsize=(10, 6))
df.boxplot(column='review_length', by='rating', grid=False, patch_artist=True)
plt.title('🔠 8. Review Length by Rating Category', fontsize=14, fontweight='bold')
plt.xlabel('Rating')
plt.ylabel('Review Length')
plt.suptitle('')
plt.show()
print("Insight:", "Unhappy users write longer reviews" if df.groupby('rating')['review_length'].mean().iloc[0] > df.groupby('rating')['review_length'].mean().iloc[-1] else "Length consistent across ratings")

In [ ]:
#9

one_star_reviews = df[df['rating'] == 1]['review'].fillna('').apply(clean_words)
all_words = ' '.join(one_star_reviews).split()
word_freq = Counter(all_words).most_common(10)
plt.figure(figsize=(12, 6))
words, counts = zip(*word_freq)
plt.bar(words, counts, color='#FF6B6B', alpha=0.8)
plt.title('💬 9. Most Mentioned Words in 1-Star Reviews', fontsize=14, fontweight='bold')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
for i, v in enumerate(counts):
    plt.text(i, v + 0.1, str(v), ha='center')
plt.tight_layout()
plt.show()
print("Top complaints:", ', '.join([w[0] for w in word_freq[:3]]))


In [ ]:
#10

version_ratings = df.groupby('version')['rating'].agg(['mean', 'count']).sort_values('mean', ascending=False)
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(version_ratings)), version_ratings['mean'], color='gold', alpha=0.8)
plt.title('📱🧪 10. Average Rating by ChatGPT Version', fontsize=14, fontweight='bold')
plt.xlabel('Version')
plt.ylabel('Average Rating')
plt.xticks(range(len(version_ratings)), version_ratings.index, rotation=45)
for i, bar in enumerate(bars):
    height = version_ratings['mean'].iloc[i]
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.2f}', ha='center')
plt.tight_layout()
plt.show()
print("Highest rated version:", version_ratings['mean'].idxmax())


In [ ]:
from sklearn.model_selection import train_test_split

# Re-split using same random state
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment']
)

# Encode AFTER split (best practice)
X_train_bert = bert_model.encode(
    X_train_text.tolist(),
    show_progress_bar=True
)

X_test_bert = bert_model.encode(
    X_test_text.tolist(),
    show_progress_bar=True)


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_bert,
    y_train
)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


def get_models():

    return {
        "Logistic Regression": LogisticRegression(
            max_iter=1000
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),

        "Neural Net (LSTM-like)": MLPClassifier(
            hidden_layer_sizes=(256,128),
            max_iter=500
        )
    }


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize


def evaluate_model(name, model, X_test, y_test):

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred))

    y_test_bin = label_binarize(
        y_test,
        classes=[0,1,2]
    )

    auc = roc_auc_score(
        y_test_bin,
        y_proba,
        multi_class="ovr"
    )

    print(f"AUC-ROC: {auc:.4f}")


In [ ]:
def train_bert_models(X_train, X_test, y_train, y_test):

    models = get_models()

    for name, model in models.items():

        print(f"\nTraining {name}...")

        model.fit(X_train, y_train)

        evaluate_model(
            name,
            model,
            X_test,
            y_test
        )


In [ ]:
train_bert_models(
    X_train_bal,
    X_test_bert,
    y_train_bal,
    y_test
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc


def plot_multiclass_roc(model, X_test, y_test, model_name):

    # Binarize labels (0,1,2)
    y_test_bin = label_binarize(y_test, classes=[0,1,2])
    n_classes = y_test_bin.shape[1]

    # Get probabilities
    y_score = model.predict_proba(X_test)

    fpr = {}
    tpr = {}
    roc_auc = {}

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Macro-average ROC
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

    mean_tpr /= n_classes

    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # Plot
    plt.figure(figsize=(8,6))

    for i in range(n_classes):
        plt.plot(
            fpr[i],
            tpr[i],
            lw=2,
            label=f"Class {i} (AUC = {roc_auc[i]:.2f})"
        )

    plt.plot(
        fpr["macro"],
        tpr["macro"],
        linestyle="--",
        color="black",
        label=f"Macro Avg (AUC = {roc_auc['macro']:.2f})"
    )

    plt.plot([0,1], [0,1], linestyle=":", color="gray")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve – {model_name}")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(256,128), max_iter=500)
}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    plot_multiclass_roc(model, X_test_bert, y_test, name)


In [ ]:
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical


def prepare_lstm_data(
    X_train,
    X_test,
    y_train,
    y_test,
    max_words=10000,
    max_len=120
):

    #
    X_train = X_train.astype(str)
    X_test = X_test.astype(str)

    # Tokenizer
    tokenizer = Tokenizer(
        num_words=max_words,
        oov_token="<OOV>"
    )

    tokenizer.fit_on_texts(X_train)

    # Text → Sequences
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)

    # Padding
    X_train_pad = pad_sequences(
        X_train_seq,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

    X_test_pad = pad_sequences(
        X_test_seq,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

    # Labels → One-hot
    y_train_cat = to_categorical(y_train, num_classes=3)
    y_test_cat = to_categorical(y_test, num_classes=3)

    return X_train_pad, X_test_pad, y_train_cat, y_test_cat, tokenizer


In [ ]:
from sklearn.utils.class_weight import compute_class_weight


def get_class_weights(y_train):

    classes = np.unique(y_train)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    return dict(zip(classes, weights))


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Input
)


def build_lstm_model(
    vocab_size,
    embedding_dim,
    max_len
):

    model = Sequential([

        # Input (IMPORTANT)
        Input(shape=(max_len,)),

        # Embedding
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim
        ),

        # Smaller BiLSTM
        Bidirectional(
            LSTM(32)
        ),

        Dropout(0.4),

        Dense(32, activation="relu"),

        Dropout(0.2),

        Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping


def train_lstm_model(
    model,
    X_train_pad,
    y_train_cat,
    class_weights,
    epochs=25,
    batch_size=16
):

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_pad,
        y_train_cat,

        validation_split=0.15,

        epochs=epochs,
        batch_size=batch_size,

        class_weight=class_weights,

        callbacks=[early_stop],

        verbose=1
    )

    return history


In [ ]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score
)

from sklearn.preprocessing import label_binarize


def evaluate_lstm_model(
    model,
    X_test_pad,
    y_test
):

    # Probabilities
    y_proba = model.predict(X_test_pad)

    # Classes
    y_pred = np.argmax(y_proba, axis=1)

    print("\n===== LSTM PERFORMANCE =====\n")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Negative","Neutral","Positive"]
        )
    )

    acc = accuracy_score(y_test, y_pred)

    print(f"Accuracy: {acc:.4f}")

    # AUC
    y_test_bin = label_binarize(
        y_test,
        classes=[0,1,2]
    )

    auc = roc_auc_score(
        y_test_bin,
        y_proba,
        multi_class="ovr"
    )

    print(f"AUC-ROC: {auc:.4f}")

    # Prediction distribution
    print("\nPrediction Distribution:")
    print(np.bincount(y_pred))


In [ ]:
def predict_reviews_lstm(
    texts,
    model,
    tokenizer,
    max_len
):

    texts = [str(t).lower() for t in texts]

    seq = tokenizer.texts_to_sequences(texts)

    pad = pad_sequences(
        seq,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

    probs = model.predict(pad)

    preds = np.argmax(probs, axis=1)

    labels = {
        0: "Negative",
        1: "Neutral",
        2: "Positive"
    }

    results = []

    for i in range(len(texts)):

        results.append({
            "Review": texts[i],
            "Prediction": labels[preds[i]],
            "Confidence": round(float(np.max(probs[i])), 3)
        })

    return results


In [ ]:
X_train_pad, X_test_pad, y_train_cat, y_test_cat, tokenizer = prepare_lstm_data(
    X_train_text,
    X_test_text,
    y_train,
    y_test
)


In [ ]:
vocab_size = min(
    10000,
    len(tokenizer.word_index) + 1
)

max_len = X_train_pad.shape[1]


In [ ]:
class_weights = get_class_weights(y_train)

print("Class Weights:", class_weights)


In [ ]:
lstm_model = build_lstm_model(
    vocab_size=vocab_size,
    embedding_dim=100,
    max_len=max_len
)

lstm_model.summary()


In [ ]:
history = train_lstm_model(
    lstm_model,
    X_train_pad,
    y_train_cat,
    class_weights
)


In [ ]:
evaluate_lstm_model(
    lstm_model,
    X_test_pad,
    y_test
)


In [ ]:
samples = [
    "This product is amazing and works perfectly",
    "It's okay, nothing special",
    "Worst experience ever. Waste of money",
    "Decent quality for the price",
    "Terrible service and bad product"
]

results = predict_reviews_lstm(
    samples,
    lstm_model,
    tokenizer,
    max_len
)

for r in results:
    print("\n", r)


In [ ]:
lstm_model.save("lstm_sentiment_model.keras")


In [ ]:
!ls


In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("lstm_sentiment_model.keras")

In [ ]:
!ls


In [ ]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)


In [ ]:
samples = [
    "This product is amazing and works perfectly",
    "It's okay, nothing special",
    "Worst experience ever. Waste of money",
    "Decent quality for the price",
    "Terrible service and bad product"
]

results = predict_reviews_lstm(
    samples,
    model,
    tokenizer,
    max_len
)

for r in results:
    print("\n", r)
